In [4]:
!pip install rasterio
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.4 MB/s eta 0:00:00


In [5]:
import gc
import json
import os
import time
from collections import defaultdict
from datetime import datetime

import cv2
import numpy as np
import psutil
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_bounds
import geopandas as gpd
from shapely.geometry import Polygon, box as shapely_box
from shapely.ops import unary_union
from tqdm import tqdm

from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [11]:
try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

# ---------------------------------------------------------------------------
MEM_LIMIT_GB = 8.0   # cache full image in RAM only if smaller than this


def _gpu_available():
    return TORCH_AVAILABLE and torch.cuda.is_available()


def setup_gpu_memory(vram_limit_gb=12.0):
    if not _gpu_available():
        print("  No GPU detected — running on CPU")
        return
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    frac     = min(vram_limit_gb / total_gb, 0.90)
    torch.cuda.set_per_process_memory_fraction(frac)
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {total_gb:.1f} GB total  →  {frac*100:.0f}% reserved "
          f"({frac*total_gb:.1f} GB)")


# ===========================================================================
# IMAGE READER
# ===========================================================================

class ImageReader:

    def __init__(self, image_path: str, mem_limit_gb: float = MEM_LIMIT_GB):
        if not os.path.exists(image_path):
            raise FileNotFoundError(image_path)
        self.image_path    = image_path
        self._cache        = None
        self._cache_done   = False
        self.original_dims = None   # (width, height)
        self.transform     = None
        self.crs           = None

        self._load_metadata()
        self._maybe_cache(mem_limit_gb)

    def _load_metadata(self):
        try:
            with rasterio.open(self.image_path) as src:
                self.original_dims = (src.width, src.height)
                self.transform     = src.transform
                self.crs           = src.crs
                self.band_count    = src.count
        except Exception:
            from PIL import Image
            with Image.open(self.image_path) as img:
                w, h = img.size
            self.original_dims = (w, h)
            self.transform     = from_bounds(0, 0, w, h, w, h)
            self.crs           = None
            self.band_count    = 3

    def _maybe_cache(self, mem_limit_gb):
        if self._cache_done:
            return
        self._cache_done = True
        w, h     = self.original_dims
        image_gb = w * h * 3 / 1024**3
        if image_gb > mem_limit_gb:
            print(f"  Image {image_gb:.1f} GB > limit — windowed reads")
            return
        print(f"  Caching {image_gb:.1f} GB image into RAM…")
        try:
            with rasterio.open(self.image_path) as src:
                bands = list(range(1, min(src.count, 3) + 1))
                if src.count >= 3:
                    data = src.read([1, 2, 3])
                    data = np.transpose(data, (1, 2, 0))
                else:
                    data = src.read(1)
                    data = cv2.cvtColor(data, cv2.COLOR_GRAY2RGB)
            self._cache = np.clip(data, 0, 255).astype(np.uint8)
            print(f"  ✓ Cached ({self._cache.nbytes/1024**3:.2f} GB)")
        except MemoryError:
            print("  MemoryError — windowed reads only")
            self._cache = None

    def read_crop(self, x1: int, y1: int, x2: int, y2: int) -> np.ndarray:
        w, h = self.original_dims
        x1 = max(0, min(x1, w));  y1 = max(0, min(y1, h))
        x2 = max(x1, min(x2, w)); y2 = max(y1, min(y2, h))
        if x2 <= x1 or y2 <= y1:
            return np.zeros((0, 0, 3), dtype=np.uint8)
        if self._cache is not None:
            return self._cache[y1:y2, x1:x2].copy()
        window = Window(x1, y1, x2 - x1, y2 - y1)
        try:
            with rasterio.open(self.image_path) as src:
                if src.count >= 3:
                    data = src.read([1, 2, 3], window=window)
                    data = np.transpose(data, (1, 2, 0))
                else:
                    data = src.read(1, window=window)
                    data = cv2.cvtColor(data, cv2.COLOR_GRAY2RGB)
            return np.clip(data, 0, 255).astype(np.uint8)
        except Exception as e:
            print(f"  ⚠ read_crop error: {e}")
            return np.zeros((y2 - y1, x2 - x1, 3), dtype=np.uint8)


# ===========================================================================
# TILED YOLO-SEG INFERENCE
# ===========================================================================

def run_yolo_seg_tiled(reader: ImageReader, model, config: dict) -> list[dict]:
    """
    Run YOLO11-seg on overlapping tiles and return a flat list of detections:
      {
        'polygon_px': np.ndarray (N,2) in original image pixel coords,
        'bbox_px':    [x1, y1, x2, y2] in original image pixel coords,
        'confidence': float,
        'class_id':   int,
        'class_name': str,
      }
    """
    tile_size   = config['tile_size']
    overlap     = config['overlap']
    resolution  = config['resolution']
    conf        = config['conf_threshold']
    iou         = config['iou_threshold']
    max_det     = config.get('max_det', 300)
    batch_size  = config.get('batch_size', 8)
    device      = 'cuda' if _gpu_available() else 'cpu'

    orig_w, orig_h = reader.original_dims
    # Scaled dimensions (the space we tile in)
    scaled_w = int(orig_w * resolution)
    scaled_h = int(orig_h * resolution)
    step      = tile_size - overlap

    # Class names from model
    class_names = model.names if hasattr(model, 'names') else {}

    tile_positions = [
        (x, y, min(x + tile_size, scaled_w), min(y + tile_size, scaled_h))
        for y in range(0, scaled_h, step)
        for x in range(0, scaled_w, step)
    ]
    print(f"  {len(tile_positions)} tiles  "
          f"(tile={tile_size}px  overlap={overlap}px  resolution={resolution})")

    all_detections = []

    for batch_start in tqdm(range(0, len(tile_positions), batch_size), desc="YOLO-seg"):
        batch    = tile_positions[batch_start : batch_start + batch_size]
        images   = []
        meta     = []   # (orig_x, orig_y, orig_x2, orig_y2, scale_x, scale_y)

        for x, y, x_end, y_end in batch:
            # Map scaled tile coords → original image coords
            ox  = int(x      / resolution)
            oy  = int(y      / resolution)
            ox2 = int(x_end  / resolution)
            oy2 = int(y_end  / resolution)
            ox  = min(ox,  orig_w - 1); oy  = min(oy,  orig_h - 1)
            ox2 = min(ox2, orig_w);     oy2 = min(oy2, orig_h)

            tile = reader.read_crop(ox, oy, ox2, oy2)
            if tile.size == 0:
                continue

            win_w, win_h = x_end - x, y_end - y
            actual_h, actual_w = tile.shape[:2]

            # Resize to tile_size for YOLO
            tile_sq = cv2.resize(tile, (tile_size, tile_size),
                                 interpolation=cv2.INTER_LINEAR)
            # Scale factors: tile_size px → original image px
            sx = actual_w / tile_size
            sy = actual_h / tile_size

            images.append(np.ascontiguousarray(tile_sq))
            meta.append((ox, oy, ox2, oy2, sx, sy))

        if not images:
            continue

        results = model.predict(
            source=images,
            conf=conf,
            iou=iou,
            max_det=max_det,
            verbose=False,
            device=device,
            retina_masks=True,   # higher-quality masks from YOLO-seg
        )

        for result, (ox, oy, ox2, oy2, sx, sy) in zip(results, meta):
            if result.masks is None or result.boxes is None:
                continue

            masks  = result.masks.data.cpu().numpy()   # (N, H_mask, W_mask)
            boxes  = result.boxes.xyxy.cpu().numpy()   # (N, 4) in tile space
            confs  = result.boxes.conf.cpu().numpy()
            cls_ids= result.boxes.cls.cpu().numpy().astype(int)

            tile_h = oy2 - oy
            tile_w = ox2 - ox

            for i in range(len(boxes)):
                # ── Mask → polygon in tile space ──────────────────────
                mask = masks[i]
                # Resize mask to original tile dimensions
                if mask.shape != (tile_h, tile_w):
                    mask = cv2.resize(
                        mask.astype(np.float32), (tile_w, tile_h),
                        interpolation=cv2.INTER_LINEAR
                    )
                binary = (mask > 0.5).astype(np.uint8)

                if binary.sum() < 9:
                    continue

                contours, _ = cv2.findContours(
                    binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
                )
                if not contours:
                    continue

                contour = max(contours, key=cv2.contourArea)
                if cv2.contourArea(contour) < 4:
                    continue

                eps    = 0.002 * cv2.arcLength(contour, True)
                approx = cv2.approxPolyDP(contour, eps, True)
                if len(approx) < 3:
                    continue

                # Polygon in tile-local coords → global original-image coords
                poly_local = approx.reshape(-1, 2).astype(np.float32)
                poly_global = poly_local + np.array([ox, oy], dtype=np.float32)

                # BBox in tile space → global
                bx1, by1, bx2, by2 = boxes[i]
                # boxes are in tile_size space; scale back to original tile dims
                bbox_global = [
                    int(bx1 * sx) + ox,
                    int(by1 * sy) + oy,
                    int(bx2 * sx) + ox,
                    int(by2 * sy) + oy,
                ]

                cid = cls_ids[i]
                all_detections.append({
                    'polygon_px': poly_global.astype(np.int32),
                    'bbox_px':    bbox_global,
                    'confidence': float(confs[i]),
                    'class_id':   cid,
                    'class_name': class_names.get(cid, str(cid)),
                })

        del images, results
        if _gpu_available() and batch_start % (batch_size * 10) == 0:
            torch.cuda.empty_cache()

    print(f"  ✓ {len(all_detections)} raw detections")
    return all_detections


# ===========================================================================
# CROSS-TILE NMS
# ===========================================================================

def polygon_nms(detections: list[dict], iou_threshold: float = 0.4) -> list[dict]:

    if not detections:
        return detections

    # Sort by confidence descending
    det = sorted(detections, key=lambda d: d['confidence'], reverse=True)

    # Build shapely polygons once
    shapes = []
    for d in det:
        try:
            p = Polygon(d['polygon_px'])
            if not p.is_valid:
                p = p.buffer(0)
            shapes.append(p)
        except Exception:
            shapes.append(None)

    keep    = []
    removed = set()

    for i in range(len(det)):
        if i in removed or shapes[i] is None:
            continue
        keep.append(det[i])
        pi   = shapes[i]
        ai   = pi.area if pi else 0

        for j in range(i + 1, len(det)):
            if j in removed or shapes[j] is None:
                continue
            pj = shapes[j]
            try:
                inter = pi.intersection(pj).area
                if inter == 0:
                    continue
                union = ai + pj.area - inter
                if union > 0 and inter / union > iou_threshold:
                    removed.add(j)
            except Exception:
                continue

    print(f"  Polygon NMS: {len(det)} → {len(keep)}  (IoU>{iou_threshold})")
    return keep


# ===========================================================================
# GEOREFERENCING
# ===========================================================================

def detections_to_geodataframe(detections: list[dict],
                                transform,
                                crs) -> gpd.GeoDataFrame:

    geo_polygons = []
    attrs        = []

    for d in tqdm(detections, desc="Georeferencing"):
        coords_px = d['polygon_px']   # (N, 2) x, y in image pixel space
        try:
            # rasterio.transform.xy expects (row, col) = (y, x)
            geo_coords = [
                rasterio.transform.xy(transform, float(py), float(px))
                for px, py in coords_px
            ]
            geo_poly = Polygon(geo_coords)
            if not geo_poly.is_valid:
                geo_poly = geo_poly.buffer(0)
            if geo_poly.is_empty:
                continue
        except Exception:
            continue

        geo_polygons.append(geo_poly)
        attrs.append({
            'confidence': d['confidence'],
            'class_id':   d['class_id'],
            'class_name': d['class_name'],
            'area_m2':    geo_poly.area,                 # in CRS units (m² if projected)
            'area_px':    Polygon(coords_px).area,
            'num_vertices': len(coords_px),
        })

    if not geo_polygons:
        return gpd.GeoDataFrame()

    gdf = gpd.GeoDataFrame(attrs, geometry=geo_polygons, crs=crs)
    print(f"  ✓ {len(gdf)} georeferenced polygons")
    return gdf


# ===========================================================================
# VISUALIZATION
# ===========================================================================

def visualize_segmentation(reader: ImageReader,
                            detections: list[dict],
                            config: dict) -> str:

    import matplotlib
    matplotlib.use('Agg')   # non-interactive backend — safe in Colab/server
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.collections import PatchCollection
    from matplotlib.patches import Polygon as MplPolygon
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors

    output_dir  = config['output_dir']
    name        = config.get('image_name', 'output')
    max_dim     = config.get('max_viz_dimension', 4000)
    dpi         = config.get('viz_dpi', 150)

    orig_w, orig_h = reader.original_dims
    scale  = min(max_dim / orig_w, max_dim / orig_h, 1.0)
    out_w  = int(orig_w * scale)
    out_h  = int(orig_h * scale)

    print(f"  Canvas: {orig_w}×{orig_h} → {out_w}×{out_h}  scale={scale:.4f}")
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, f"{name}_segmentation.png")

    # ── Build downsampled background ──────────────────────────────────
    # Read in strips to stay memory-safe, then assemble
    strip_src_h = config.get('viz_strip_height', 512)
    bg = np.zeros((out_h, out_w, 3), dtype=np.uint8)

    for y0 in range(0, out_h, max(1, int(strip_src_h * scale))):
        y1     = min(y0 + max(1, int(strip_src_h * scale)), out_h)
        src_y0 = int(y0 / scale)
        src_y1 = int(y1 / scale)
        src    = reader.read_crop(0, src_y0, orig_w, src_y1)
        if src.size == 0: continue
        bg[y0:y1] = cv2.resize(src, (out_w, y1 - y0), interpolation=cv2.INTER_AREA)

    # ── Figure setup ──────────────────────────────────────────────────
    fig_w = out_w / dpi
    fig_h = out_h / dpi
    fig, ax = plt.subplots(figsize=(fig_w + 1.4, fig_h))   # +1.4 for colorbar

    ax.imshow(bg / 255.0, extent=[0, out_w, out_h, 0], aspect='equal', alpha=0.85)
    ax.set_xlim(0, out_w)
    ax.set_ylim(out_h, 0)
    ax.set_axis_off()
    ax.set_title("Crop Field Detections", fontsize=14, fontweight='bold', pad=10)

    # ── Draw polygons coloured by confidence (plasma colormap) ────────
    cmap  = cm.plasma
    norm  = mcolors.Normalize(vmin=0.0, vmax=1.0)

    patch_list  = []
    conf_values = []

    for d in detections:
        pts = (d['polygon_px'] * scale).astype(np.float32)
        pts[:, 0] = np.clip(pts[:, 0], 0, out_w - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, out_h - 1)
        if len(pts) < 3:
            continue
        patch_list.append(MplPolygon(pts, closed=True))
        conf_values.append(d['confidence'])

    if patch_list:
        face_colors = [cmap(norm(c)) for c in conf_values]
        # Semi-transparent fill
        fill_colors = [(r, g, b, 0.25) for r, g, b, _ in face_colors]

        pc = PatchCollection(
            patch_list,
            facecolors=fill_colors,
            edgecolors=[(1.0, 0.0, 0.0, 0.9)] * len(patch_list),  # red outlines
            linewidths=1.2,
        )
        ax.add_collection(pc)

    # ── Colorbar ──────────────────────────────────────────────────────
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.7, pad=0.02)
    cbar.set_label('Confidence', fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    # ── Stats box (top-left, matching reference style) ─────────────────
    n     = len(detections)
    avg_c = np.mean(conf_values) if conf_values else 0.0

    stats_lines = [f"Total: {n}", f"Avg Conf: {avg_c:.3f}"]
    # Add per-class counts if multi-class
    class_counts: dict[str, int] = {}
    for d in detections:
        class_counts[d['class_name']] = class_counts.get(d['class_name'], 0) + 1
    if len(class_counts) > 1:
        for cname, cnt in sorted(class_counts.items()):
            stats_lines.append(f"{cname}: {cnt}")

    stats_text = "\n".join(stats_lines)
    ax.text(
        0.02, 0.98, stats_text,
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment='top',
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9,
                  edgecolor="black", linewidth=0.8),
        fontfamily='monospace',
    )

    # ── Save ──────────────────────────────────────────────────────────
    plt.tight_layout(pad=0.5)
    plt.savefig(out_path, dpi=dpi, bbox_inches='tight',
                pil_kwargs={'compress_level': 6})
    plt.close(fig)

    del bg, patch_list
    gc.collect()

    print(f"  ✓ PNG: {out_path}  ({os.path.getsize(out_path)/1024**2:.1f} MB)")
    return out_path


# ===========================================================================
# SAVE OUTPUTS
# ===========================================================================

def save_outputs(gdf: gpd.GeoDataFrame,
                 detections: list[dict],
                 config: dict,
                 elapsed_sec: float):
    output_dir = config['output_dir']
    name       = config.get('image_name', 'output')
    os.makedirs(output_dir, exist_ok=True)

    # GeoJSON
    if gdf is not None and len(gdf) > 0 and gdf.crs is not None:
        p = os.path.join(output_dir, f"{name}_fields.geojson")
        try:
            gdf.to_file(p, driver='GeoJSON')
            print(f"  ✓ GeoJSON : {p}")
        except Exception as e:
            print(f"  ⚠ GeoJSON failed: {e}")

    # CSV (no geometry)
    if gdf is not None and len(gdf) > 0:
        p = os.path.join(output_dir, f"{name}_fields.csv")
        try:
            gdf.drop(columns='geometry').to_csv(p, index=False)
            print(f"  ✓ CSV     : {p}")
        except Exception as e:
            print(f"  ⚠ CSV failed: {e}")

    # Statistics JSON
    if detections:
        confs = [d['confidence'] for d in detections]
        areas = [Polygon(d['polygon_px']).area for d in detections
                 if len(d['polygon_px']) >= 3]
        class_counts: dict[str, int] = {}
        for d in detections:
            class_counts[d['class_name']] = class_counts.get(d['class_name'], 0) + 1

        stats = {
            'total_fields':          len(detections),
            'class_counts':          class_counts,
            'avg_confidence':        float(np.mean(confs)),
            'min_confidence':        float(np.min(confs)),
            'max_confidence':        float(np.max(confs)),
            'avg_area_px':           float(np.mean(areas)) if areas else 0,
            'median_area_px':        float(np.median(areas)) if areas else 0,
            'processing_time_min':   round(elapsed_sec / 60, 2),
        }
        p = os.path.join(output_dir, f"{name}_statistics.json")
        try:
            with open(p, 'w') as f:
                json.dump(stats, f, indent=2)
            print(f"  ✓ Stats   : {p}")
        except Exception as e:
            print(f"  ⚠ Stats failed: {e}")


# ===========================================================================
# MAIN PIPELINE
# ===========================================================================

def run_cropfield_segmentation(image_path: str,
                                model,
                                config: dict):
    """
    Full pipeline:
      1. Load image metadata + optional RAM cache
      2. Tiled YOLO11-seg inference
      3. Cross-tile polygon NMS
      4. Georeferencing → GeoDataFrame
      5. Visualization
      6. Save GeoJSON, CSV, statistics
    """
    t0      = time.time()
    process = psutil.Process()

    def _ram():
        return process.memory_info().rss / 1024**3

    print(f"\n{'='*65}")
    print("  CROP FIELD SEGMENTATION PIPELINE")
    print(f"{'='*65}")
    print(f"  Image      : {image_path}")
    print(f"  Model      : {config.get('model_path', 'provided externally')}")
    print(f"  Output dir : {config['output_dir']}")

    # ── 1. Image reader ───────────────────────────────────────────────
    print(f"\n[1/5] Loading image…")
    reader = ImageReader(image_path)
    orig_w, orig_h = reader.original_dims
    print(f"  ✓ {orig_w}×{orig_h}  bands={reader.band_count}  "
          f"CRS={reader.crs}  RAM={_ram():.2f} GB")

    # ── 2. Tiled YOLO-seg ─────────────────────────────────────────────
    print(f"\n[2/5] Running YOLO11-seg…")
    detections = run_yolo_seg_tiled(reader, model, config)
    print(f"  RAM: {_ram():.2f} GB  |  t={time.time()-t0:.1f}s")

    if not detections:
        print("  ⚠ No fields detected. Check conf_threshold and model path.")
        return None, []

    # ── 3. Cross-tile polygon NMS ─────────────────────────────────────
    print(f"\n[3/5] Polygon NMS…")
    nms_iou    = config.get('polygon_nms_iou', 0.4)
    detections = polygon_nms(detections, nms_iou)
    print(f"  RAM: {_ram():.2f} GB  |  t={time.time()-t0:.1f}s")

    # ── 4. Georeference ───────────────────────────────────────────────
    print(f"\n[4/5] Georeferencing…")
    gdf = detections_to_geodataframe(detections, reader.transform, reader.crs)
    print(f"  RAM: {_ram():.2f} GB  |  t={time.time()-t0:.1f}s")

    # ── 5. Visualization ──────────────────────────────────────────────
    print(f"\n[5/5] Visualization…")
    visualize_segmentation(reader, detections, config)

    # ── Save ──────────────────────────────────────────────────────────
    elapsed = time.time() - t0
    print(f"\n[Save] Writing outputs…")
    save_outputs(gdf, detections, config, elapsed)

    print(f"\n{'='*65}")
    print(f"  ✓ Done in {elapsed/60:.1f} min  |  {len(detections)} fields")
    print(f"  Peak RAM: {_ram():.2f} GB")
    print(f"{'='*65}\n")

    return gdf, detections

# ===========================================================================
# ENTRY POINT
# ===========================================================================

if __name__ == "__main__":
    import torch
    from ultralytics import YOLO

    # Enable cuDNN autotuner + medium matmul precision
    torch.backends.cudnn.benchmark       = True
    torch.set_float32_matmul_precision('medium')

    setup_gpu_memory(vram_limit_gb=12.0)

    # ── Load model ────────────────────────────────────────────────────
    MODEL_PATH = (
        '/content/drive/MyDrive/AGRI/CropField_Segmentation/result/'
        'yolov11n-seg-cropfield-100epoch-version01/weights/best.pt'
    )
    model = YOLO(MODEL_PATH)
    model.fuse()
    if torch.cuda.is_available():
        model.half()   # FP16 — faster on T4, lower VRAM

    # ── Config ────────────────────────────────────────────────────────
    config = {
        # Tiling
        'tile_size':        1280,   # px fed to YOLO (must match training imgsz)
        'overlap':          128,    # px overlap between tiles (catches edge fields)
        'resolution':       1.0,    # downsample factor (1.0 = full res)
        'batch_size':       8,      # tiles per YOLO batch

        # Detection thresholds
        'conf_threshold':   0.25,
        'iou_threshold':    0.30,   # YOLO internal NMS
        'max_det':          1000,
        'polygon_nms_iou':  0.20,   # cross-tile polygon dedup

        # Visualization
        'max_viz_dimension': 4000,
        'viz_strip_height':  512,

        # Output
        'output_dir':  '/content/drive/MyDrive/AGRI/CropField_Segmentation/output',
        'image_name':  'cropfield_seg',
        'model_path':  MODEL_PATH,
    }

    IMAGE_PATH = '/content/drive/MyDrive/AGRI/CropField_Segmentation/Screenshot 2026-03-17 063235.png'

    gdf, detections = run_cropfield_segmentation(IMAGE_PATH, model, config)

  GPU  : Tesla T4
  VRAM : 14.6 GB total  →  82% reserved (12.0 GB)
YOLO11n-seg summary (fused): 114 layers, 2,834,763 parameters, 0 gradients, 9.6 GFLOPs

  CROP FIELD SEGMENTATION PIPELINE
  Image      : /content/drive/MyDrive/AGRI/CropField_Segmentation/Screenshot 2026-03-17 063235.png
  Model      : /content/drive/MyDrive/AGRI/CropField_Segmentation/result/yolov11n-seg-cropfield-100epoch-version01/weights/best.pt
  Output dir : /content/drive/MyDrive/AGRI/CropField_Segmentation/output

[1/5] Loading image…
  Caching 0.0 GB image into RAM…


/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


  ✓ Cached (0.00 GB)
  ✓ 1006×768  bands=4  CRS=None  RAM=1.64 GB

[2/5] Running YOLO11-seg…
  1 tiles  (tile=1280px  overlap=128px  resolution=1.0)


YOLO-seg: 100%|██████████| 1/1 [00:00<00:00,  4.94it/s]


  ✓ 33 raw detections
  RAM: 1.65 GB  |  t=0.2s

[3/5] Polygon NMS…
  Polygon NMS: 33 → 32  (IoU>0.2)
  RAM: 1.65 GB  |  t=0.3s

[4/5] Georeferencing…


Georeferencing: 100%|██████████| 32/32 [00:00<00:00, 757.87it/s]


  ✓ 32 georeferenced polygons
  RAM: 1.65 GB  |  t=0.3s

[5/5] Visualization…
  Canvas: 1006×768 → 1006×768  scale=1.0000
  ✓ PNG: /content/drive/MyDrive/AGRI/CropField_Segmentation/output/cropfield_seg_segmentation.png  (1.1 MB)

[Save] Writing outputs…
  ✓ CSV     : /content/drive/MyDrive/AGRI/CropField_Segmentation/output/cropfield_seg_fields.csv
  ✓ Stats   : /content/drive/MyDrive/AGRI/CropField_Segmentation/output/cropfield_seg_statistics.json

  ✓ Done in 0.0 min  |  32 fields
  Peak RAM: 1.66 GB

